In [3]:
# day2_fetch_sequences.py

import pandas as pd
import time
import random
import os
from Bio import Entrez, SeqIO

# ─────────────────────────────────────────────
# CONFIGURATION — edit these values
# ─────────────────────────────────────────────

Entrez.email = "rminahmdi2000@gmail.com"   # NCBI requires this — use your real email
Entrez.api_key = "2f3265e9536b4022153686e85db1ce6ad008"      # Paste your NCBI API key here

INPUT_CSV  = "filtered_variant_summary.csv"
OUTPUT_CSV = "sequences_dataset.csv"
CHECKPOINT = "checkpoint.csv"             # Tracks progress so we can resume

WINDOW     = 256    # nucleotides on EACH side of the variant
# Total sequence length = WINDOW * 2 = 512 bp

SAMPLE_PER_CLASS = 5000   # How many Pathogenic and Benign sequences to fetch
REQUESTS_PER_SEC = 8      # Stay safely below the 10/sec limit

# ─────────────────────────────────────────────
# STEP 1: Load your filtered CSV and sample it
# ─────────────────────────────────────────────

df = pd.read_csv(INPUT_CSV)

# WHY: We need to know exactly which string labels your CSV uses.
# Day 1 filtering might have kept 'Pathogenic' or 'Pathogenic/Likely pathogenic'.
# Check what you have:
print("Label distribution in your CSV:")
print(df['ClinicalSignificance'].value_counts())
print()

# Separate into two groups
# WHY: We sample each class independently to guarantee equal numbers.
# If we sampled the whole df at once, imbalance would carry through.
pathogenic = df[df['ClinicalSignificance'].str.contains('Pathogenic', case=False, na=False)]
benign     = df[df['ClinicalSignificance'].str.contains('Benign', case=False, na=False)]

# Remove any overlap — entries that say "Pathogenic/Likely benign" (edge cases)
# Keep strict labels only for cleaner training signal
pathogenic = pathogenic[~pathogenic['ClinicalSignificance'].str.contains('Benign', case=False, na=False)]
benign     = benign[~benign['ClinicalSignificance'].str.contains('Pathogenic', case=False, na=False)]

print(f"Strict Pathogenic entries available: {len(pathogenic)}")
print(f"Strict Benign entries available:     {len(benign)}")

# Sample equally from each
# random_state=42 means the same sample every time you run — reproducibility
sample_path = pathogenic.sample(n=min(SAMPLE_PER_CLASS, len(pathogenic)), random_state=42)
sample_ben  = benign.sample(n=min(SAMPLE_PER_CLASS, len(benign)), random_state=42)

# Combine and shuffle
to_fetch = pd.concat([sample_path, sample_ben]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"\nTotal variants to fetch sequences for: {len(to_fetch)}")

# ─────────────────────────────────────────────
# STEP 2: Resume logic
# ─────────────────────────────────────────────
# WHY: If this script runs for hours and something crashes, we don't
# want to restart from zero. We save progress to a checkpoint file
# and skip variants we've already fetched.

if os.path.exists(CHECKPOINT):
    done = pd.read_csv(CHECKPOINT)
    done_ids = set(done['VariationID'].astype(str))
    print(f"Resuming — {len(done)} sequences already fetched, skipping those.")
else:
    done = pd.DataFrame()
    done_ids = set()

# ─────────────────────────────────────────────
# STEP 3: The fetch function
# ─────────────────────────────────────────────
# WHY: This is the core operation — given a chromosome and position,
# ask NCBI for the DNA sequence in a window around that position.

def fetch_sequence(chrom, start, window=256, retries=3):
    """
    Fetches a DNA sequence window from NCBI's human genome (GRCh38).

    Parameters:
        chrom  : chromosome number/name, e.g. '17' or 'X'
        start  : the variant's position in the genome
        window : how many bp to grab on each side
        retries: how many times to retry on failure

    Returns:
        A string of nucleotides (e.g. 'ATCG...'), or None if fetch failed.
    """

    # Build the chromosome accession number for GRCh38
    # WHY: NCBI doesn't use "Chr17" — it uses accession numbers like "NC_000017.11"
    # This dictionary maps plain chromosome names to their GRCh38 accessions
    chrom_map = {
        '1': 'NC_000001.11', '2': 'NC_000002.12', '3': 'NC_000003.12',
        '4': 'NC_000004.12', '5': 'NC_000005.10', '6': 'NC_000006.12',
        '7': 'NC_000007.14', '8': 'NC_000008.11', '9': 'NC_000009.12',
        '10': 'NC_000010.11', '11': 'NC_000011.10', '12': 'NC_000012.12',
        '13': 'NC_000013.11', '14': 'NC_000014.9',  '15': 'NC_000015.10',
        '16': 'NC_000016.10', '17': 'NC_000017.11', '18': 'NC_000018.10',
        '19': 'NC_000019.10', '20': 'NC_000020.11', '21': 'NC_000021.9',
        '22': 'NC_000022.11', 'X':  'NC_000023.11', 'Y':  'NC_000024.10'
    }

    chrom = str(chrom).replace('chr', '').replace('Chr', '')

    if chrom not in chrom_map:
        return None   # Skip mitochondrial (MT) or unrecognized chromosomes

    accession = chrom_map[chrom]

    # Calculate the window boundaries
    seq_start = max(1, start - window)    # Never go below position 1
    seq_end   = start + window

    # Try fetching, with retries on failure
    for attempt in range(retries):
        try:
            # efetch is the Entrez function for fetching a specific record
            # db='nuccore' = nucleotide core database (where genome sequences live)
            # rettype='fasta' = give me the sequence in FASTA format
            handle = Entrez.efetch(
                db       = 'nuccore',
                id       = accession,
                rettype  = 'fasta',
                retmode  = 'text',
                seq_start = seq_start,
                seq_stop  = seq_end
            )
            record = SeqIO.read(handle, 'fasta')
            handle.close()
            return str(record.seq).upper()   # Return as uppercase string

        except Exception as e:
            # WHY exponential backoff: if NCBI is struggling, hammering it
            # with immediate retries makes things worse. Wait longer each time.
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"  Attempt {attempt+1} failed for Chr{chrom}:{start} — {e}. Retrying in {wait:.1f}s...")
            time.sleep(wait)

    return None   # All retries exhausted

Label distribution in your CSV:
ClinicalSignificance
Benign        93525
Pathogenic    41484
Name: count, dtype: int64

Strict Pathogenic entries available: 41484
Strict Benign entries available:     93525

Total variants to fetch sequences for: 10000


In [7]:
# ─────────────────────────────────────────────
# STEP 4: Main fetch loop
# ─────────────────────────────────────────────

results = []
delay   = 1.0 / REQUESTS_PER_SEC   # Seconds to wait between each request

print("\nStarting sequence fetch...\n")

for i, row in to_fetch.iterrows():

    var_id = str(row['Name']) # Changed from VariationID to Name

    # Skip if already fetched (resume logic)
    if var_id in done_ids:
        continue

    # Assign numeric label
    # WHY: ML models need numbers, not strings
    is_pathogenic = 'pathogenic' in str(row['ClinicalSignificance']).lower()
    label = 1 if is_pathogenic else 0

    # Fetch the sequence
    seq = fetch_sequence(
        chrom  = row['Chromosome'],
        start  = int(row['Start']),
        window = WINDOW
    )

    if seq is not None:
        results.append({
            'VariationID'          : var_id, # Keep this key name for output, assign from 'Name'
            'sequence'             : seq,
            'label'                : label,
            'ClinicalSignificance' : row['ClinicalSignificance'],
            'Chromosome'           : row['Chromosome'],
            'Start'                : row['Start']
        })
    else:
        print(f"  ⚠ Failed to fetch Chr{row['Chromosome']}:{row['Start']} — skipping")

    # Progress report every 100 variants
    if len(results) % 100 == 0 and len(results) > 0:
        print(f"  ✓ {len(results) + len(done)} sequences fetched so far...")

        # Save checkpoint — append new results to checkpoint file
        checkpoint_df = pd.DataFrame(results)
        if os.path.exists(CHECKPOINT):
            checkpoint_df.to_csv(CHECKPOINT, mode='a', header=False, index=False)
        else:
            checkpoint_df.to_csv(CHECKPOINT, index=False)

        # Add to done_ids so we don't double-save
        done_ids.update(checkpoint_df['VariationID'].astype(str))
        results = []   # Clear buffer after saving

    # Rate limiting — the most important single line in this script
    time.sleep(delay)


Starting sequence fetch...

  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ⚠ Failed to fetch ChrMT:4012 — skipping
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so far...
  ✓ 100 sequences fetched so 

In [8]:
# ─────────────────────────────────────────────
# STEP 5: Save final output
# ─────────────────────────────────────────────

# Save any remaining results not yet checkpointed
if results:
    checkpoint_df = pd.DataFrame(results)
    if os.path.exists(CHECKPOINT):
        checkpoint_df.to_csv(CHECKPOINT, mode='a', header=False, index=False)
    else:
        checkpoint_df.to_csv(CHECKPOINT, index=False)

# Combine checkpoint with any previously done rows and save final file
# Assuming checkpoint_df now has 'VariationID' due to changes in STEP 4
final = pd.read_csv(CHECKPOINT)
final = final.drop_duplicates(subset='VariationID')

final.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Done! {len(final)} sequences saved to '{OUTPUT_CSV}'")
print(f"Label distribution:")
print(final['label'].value_counts())


✅ Done! 9995 sequences saved to 'sequences_dataset.csv'
Label distribution:
label
1    4999
0    4996
Name: count, dtype: int64


**Labels:**

1: pathogenic

2: benign

so the balance is perfect, and the only goal for the third day is to make the dataset is trustworthy before I hand it to a model.